# Real-World A/B Testing with Binomial GLMs

This notebook demonstrates how to use the `ab-glm-abtest` package for analyzing a real A/B test with:
- Clustered data (multiple sessions per user)
- Covariate adjustment for increased power
- Proper statistical inference with cluster-robust standard errors
- Business-friendly metrics (ATE and Risk Ratio)

## Scenario

Imagine we're running an e-commerce A/B test where:
- **Treatment**: New checkout flow designed to reduce friction
- **Control**: Existing checkout flow
- **Outcome**: Binary conversion (1 = purchase, 0 = no purchase)
- **Unit of randomization**: User (not session)
- **Covariates**: Country (EU/non-EU), device type, prior engagement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ab_glm import (
    simulate_ab_data,
    fit_binomial_glm,
    marginal_effects_ate_and_rr,
    brier_score,
    ABResults
)

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load and Explore Your Data

In practice, you would load your actual experiment data. Here we'll use simulated data that mimics real-world patterns.

In [ ]:
# Option 1: Load your real data
# df = pd.read_csv('../examples/sample_experiment_data.csv')

# Option 2: Use simulated data for demonstration
df = simulate_ab_data(n_users=5000, sessions_per_user=(1, 7), seed=42)

print(f"Dataset shape: {df.shape}")
print(f"Unique users: {df['user_id'].nunique():,}")
print(f"Total sessions: {len(df):,}")
print(f"\nFirst few rows:")
df.head(10)

## 2. Data Quality Checks

Before analysis, always verify data quality and randomization balance.

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print()

# Verify treatment is assigned at user level (not session level)
treatment_consistency = df.groupby('user_id')['T'].nunique()
assert (treatment_consistency == 1).all(), "Treatment varies within users!"
print("✓ Treatment assignment is consistent at user level")

# Check randomization balance
user_level = df.groupby('user_id').first()
balance = user_level.groupby('T')[['country_EU', 'device_mobile', 'prior_views']].mean()
print("\nCovariate balance by treatment group:")
print(balance)

# Sessions per user distribution
sessions_per_user = df.groupby('user_id').size()
print(f"\nSessions per user - Mean: {sessions_per_user.mean():.2f}, Median: {sessions_per_user.median():.0f}")

## 3. Exploratory Data Analysis

In [ ]:
# Overall conversion rates
print("Raw conversion rates:")
print(f"Control: {df[df['T']==0]['y'].mean():.4f}")
print(f"Treatment: {df[df['T']==1]['y'].mean():.4f}")
print(f"Raw lift: {(df[df['T']==1]['y'].mean() - df[df['T']==0]['y'].mean()):.4f}")

# Conversion by covariates
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# By treatment
conv_by_treatment = df.groupby('T')['y'].mean()
axes[0].bar(['Control', 'Treatment'], conv_by_treatment.values, color=['#3498db', '#e74c3c'])
axes[0].set_ylabel('Conversion Rate')
axes[0].set_title('Conversion by Treatment')
axes[0].set_ylim(0, conv_by_treatment.max() * 1.2)

# By device
conv_by_device = df.groupby(['device_mobile', 'T'])['y'].mean().unstack()
conv_by_device.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'])
axes[1].set_xticklabels(['Desktop', 'Mobile'], rotation=0)
axes[1].set_ylabel('Conversion Rate')
axes[1].set_title('Conversion by Device & Treatment')
axes[1].legend(['Control', 'Treatment'])

# By country
conv_by_country = df.groupby(['country_EU', 'T'])['y'].mean().unstack()
conv_by_country.plot(kind='bar', ax=axes[2], color=['#3498db', '#e74c3c'])
axes[2].set_xticklabels(['Non-EU', 'EU'], rotation=0)
axes[2].set_ylabel('Conversion Rate')
axes[2].set_title('Conversion by Region & Treatment')
axes[2].legend(['Control', 'Treatment'])

plt.tight_layout()
plt.show()

## 4. Fit Binomial GLM with Logit Link

Now we fit the main statistical model with:
- Covariate adjustment for improved precision
- Cluster-robust standard errors at the user level

In [ ]:
# Fit the model
glm, _, df_model, results = fit_binomial_glm(df, link="logit", cluster_col="user_id")

print("="*50)
print("BINOMIAL GLM RESULTS (LOGIT LINK)")
print("="*50)
print(f"\nNumber of observations: {len(df_model):,}")
print(f"Number of clusters (users): {df_model['user_id'].nunique():,}")
print(f"\nModel formula: y ~ T + country_EU + device_mobile + prior_views")
print(f"\nCoefficients (with cluster-robust SEs):")
print(results.summary2().tables[1])

## 5. Compute Business-Friendly Metrics

Convert model coefficients to interpretable metrics:
- **ATE (Average Treatment Effect)**: Absolute change in conversion probability
- **Risk Ratio**: Relative change (treated/control)

In [ ]:
# Calculate marginal effects
ate_rd, rr, p_treated, p_control = marginal_effects_ate_and_rr(results, df_model)

# Extract treatment coefficient and SE
treat_coef = results.params['T']
treat_se = np.sqrt(np.diag(results.cov_params()))[list(results.params.index).index('T')]
z_score = treat_coef / treat_se
p_value = 2 * (1 - np.abs(np.random.standard_normal(10000) < np.abs(z_score)).mean())

# Calculate confidence intervals for ATE
# Using delta method approximation
ate_se_approx = treat_se * p_control * (1 - p_control)  # Simplified approximation
ate_ci_lower = ate_rd - 1.96 * ate_se_approx
ate_ci_upper = ate_rd + 1.96 * ate_se_approx

print("="*50)
print("TREATMENT EFFECT ESTIMATES")
print("="*50)
print(f"\nCovariate-Adjusted Estimates:")
print(f"Control conversion rate: {p_control:.4f} ({p_control*100:.2f}%)")
print(f"Treatment conversion rate: {p_treated:.4f} ({p_treated*100:.2f}%)")
print(f"\nAbsolute Treatment Effect (ATE):")
print(f"  Point estimate: {ate_rd:.4f} ({ate_rd*100:.2f} percentage points)")
print(f"  95% CI: [{ate_ci_lower:.4f}, {ate_ci_upper:.4f}]")
print(f"\nRelative Treatment Effect:")
print(f"  Risk Ratio: {rr:.4f}")
print(f"  Relative lift: {(rr - 1)*100:.2f}%")
print(f"\nStatistical Significance:")
print(f"  Z-score: {z_score:.3f}")
print(f"  P-value: {p_value:.4f}")
print(f"  Significant at α=0.05: {'Yes ✓' if p_value < 0.05 else 'No ✗'}")

## 6. Model Diagnostics

In [ ]:
# Calculate in-sample predictions
predictions = results.predict(df_model)

# Brier score (calibration metric)
bs = brier_score(df_model['y'].values, predictions)
print(f"Brier Score: {bs:.4f}")
print("(Lower is better; 0 = perfect, 0.25 = random guessing)\n")

# Plot calibration
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Calibration plot
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

observed_freq = []
predicted_freq = []
bin_counts = []

for i in range(n_bins):
    mask = (predictions >= bin_edges[i]) & (predictions < bin_edges[i+1])
    if mask.sum() > 0:
        observed_freq.append(df_model.loc[mask, 'y'].mean())
        predicted_freq.append(predictions[mask].mean())
        bin_counts.append(mask.sum())
    else:
        observed_freq.append(np.nan)
        predicted_freq.append(np.nan)
        bin_counts.append(0)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
axes[0].scatter(predicted_freq, observed_freq, s=[c/10 for c in bin_counts], alpha=0.7)
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Observed Frequency')
axes[0].set_title(f'Calibration Plot (Brier Score: {bs:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution of predictions by treatment
pred_control = predictions[df_model['T'] == 0]
pred_treated = predictions[df_model['T'] == 1]

axes[1].hist(pred_control, bins=30, alpha=0.5, label='Control', color='#3498db', density=True)
axes[1].hist(pred_treated, bins=30, alpha=0.5, label='Treatment', color='#e74c3c', density=True)
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution of Predicted Probabilities')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Heterogeneous Treatment Effects

Explore whether treatment effects vary by user characteristics.

In [ ]:
# Calculate treatment effects by subgroup
subgroups = {
    'device_mobile': ['Desktop', 'Mobile'],
    'country_EU': ['Non-EU', 'EU']
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (var, labels) in enumerate(subgroups.items()):
    ate_by_group = []
    
    for val in [0, 1]:
        subset = df_model[df_model[var] == val].copy()
        if len(subset) > 0:
            # Recompute ATE for this subgroup
            subset_copy = subset.copy()
            subset_copy['T'] = 1
            p1 = results.predict(subset_copy).mean()
            subset_copy['T'] = 0
            p0 = results.predict(subset_copy).mean()
            ate = p1 - p0
            ate_by_group.append(ate)
        else:
            ate_by_group.append(0)
    
    axes[idx].bar(labels, [a * 100 for a in ate_by_group], color=['#95a5a6', '#34495e'])
    axes[idx].set_ylabel('ATE (percentage points)')
    axes[idx].set_title(f'Treatment Effect by {var.replace("_", " ").title()}')
    axes[idx].axhline(y=ate_rd * 100, color='red', linestyle='--', alpha=0.5, label='Overall ATE')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: These are descriptive comparisons. For formal heterogeneity testing,")
print("include interaction terms (T × covariate) in the model.")

## 8. Business Recommendations

Based on the analysis, we can make data-driven recommendations.

In [ ]:
# Calculate practical significance
baseline_revenue = 1000000  # Example: $1M monthly revenue
revenue_lift = baseline_revenue * ate_rd

print("="*50)
print("BUSINESS IMPACT SUMMARY")
print("="*50)

if p_value < 0.05 and ate_rd > 0:
    print("✅ RECOMMENDATION: Ship the new checkout flow\n")
    print(f"Expected impact:")
    print(f"  • Absolute conversion lift: +{ate_rd*100:.2f} percentage points")
    print(f"  • Relative conversion lift: +{(rr-1)*100:.1f}%")
    print(f"  • Estimated revenue increase: ${revenue_lift:,.0f}/month")
    print(f"\nConfidence: High (p-value: {p_value:.4f})")
elif p_value < 0.05 and ate_rd < 0:
    print("⚠️ RECOMMENDATION: Do NOT ship the new checkout flow\n")
    print(f"Negative impact detected:")
    print(f"  • Absolute conversion drop: {ate_rd*100:.2f} percentage points")
    print(f"  • Relative conversion drop: {(rr-1)*100:.1f}%")
    print(f"  • Estimated revenue loss: ${abs(revenue_lift):,.0f}/month")
else:
    print("🔍 RECOMMENDATION: Continue testing\n")
    print(f"Current results:")
    print(f"  • Effect size: {ate_rd*100:.2f} percentage points")
    print(f"  • P-value: {p_value:.4f} (not significant at α=0.05)")
    print(f"  • Consider: Increasing sample size or test duration")

# Sample size recommendation for future tests
from scipy.stats import norm
alpha = 0.05
power = 0.80
mde = 0.01  # Minimum detectable effect of 1 percentage point

# Simplified sample size calculation
z_alpha = norm.ppf(1 - alpha/2)
z_beta = norm.ppf(power)
p_pooled = (p_control + p_treated) / 2
n_per_group = 2 * p_pooled * (1 - p_pooled) * ((z_alpha + z_beta) / mde) ** 2

print(f"\n" + "="*50)
print("FUTURE TEST PLANNING")
print("="*50)
print(f"For detecting a {mde*100:.1f} percentage point difference:")
print(f"  • Required sample size: {int(n_per_group):,} users per group")
print(f"  • Total users needed: {int(2*n_per_group):,}")
print(f"  • Current test had: {df_model['user_id'].nunique():,} users")

## 9. Export Results

Save the results for reporting and documentation.

In [ ]:
# Create results summary
results_summary = {
    'metric': [
        'Sample Size (users)',
        'Sample Size (sessions)',
        'Control Conversion Rate',
        'Treatment Conversion Rate',
        'Absolute Treatment Effect (ATE)',
        'Risk Ratio',
        'Relative Lift (%)',
        'P-value',
        'Significant at α=0.05',
        'Brier Score'
    ],
    'value': [
        df_model['user_id'].nunique(),
        len(df_model),
        f"{p_control:.4f}",
        f"{p_treated:.4f}",
        f"{ate_rd:.4f}",
        f"{rr:.4f}",
        f"{(rr-1)*100:.2f}",
        f"{p_value:.4f}",
        'Yes' if p_value < 0.05 else 'No',
        f"{bs:.4f}"
    ]
}

results_df = pd.DataFrame(results_summary)
print("Results Summary:")
print(results_df.to_string(index=False))

# Save to CSV
# results_df.to_csv('ab_test_results.csv', index=False)
# print("\nResults saved to 'ab_test_results.csv'")

## Key Takeaways

1. **Covariate adjustment** improves precision by accounting for pre-treatment characteristics
2. **Cluster-robust SEs** provide valid inference when users have multiple sessions
3. **Marginal effects** (ATE and RR) translate model coefficients to business metrics
4. **Model diagnostics** (Brier score, calibration plots) ensure reliable predictions
5. **Heterogeneity analysis** can reveal important subgroup differences

## Next Steps

- For your own data, replace the simulated data with your CSV file
- Consider adding interaction terms if you suspect heterogeneous effects
- Use the probit link as a robustness check
- Implement sequential testing if monitoring the test over time